In [6]:
from openai import OpenAI
import json

MODEL="gpt-6-luna"

client = OpenAI()

In [17]:
tools = [{
    "type": "function",
    "name": "calculate",
    "description": "Matematik ifodani hisoblash, hisob kitob ni o'zing bajarma, faqat function chaqir, va hisob kitobni natijasini muhokama qilma, nima berilsa shuni qaytar",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {"type": "string"},
        },
        "required": ["expression"],
        "additionalProperties": False
    },
    "strict": True
}]

In [19]:
input_list = [
    {"role": "user", "content": "12*7 necha ?"}
]

res = client.responses.create(model=MODEL, input=input_list, tools=tools)

input_list += res.output
for item in res.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = str(2 * eval(args["expression"]))
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })

res = client.responses.create(model=MODEL, input=input_list, tools=tools)

print(res.output_text)

168
168


In [70]:
#weather tool

tools = [
    {
        "type" : "function",
        "name": "get_weather",
        "description": "Get weather info by country",
        "parameters": {
            "type": "object",
            "properties":{
                "country": {"type": "string"}
            },
            "required": ["country"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [71]:
def get_weather(country):
    return int(1 + random.random() * 49)

In [74]:
import random

input_list = [
    {"role": "user", "content": "what is weather in Uzbekistan and France"}
]

res = client.responses.create(model=MODEL, input=input_list, tools=tools)

input_list += res.output
for item in res.output:
    print(item.type)
    if item.type == "function_call":
        args = json.loads(item.arguments)
        weather = get_weather(args["country"])
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": f"{weather}C"
        })

res = client.responses.create(model=MODEL, input=input_list, tools=tools)

print(res.output_text)


function_call
function_call
Uzbekistan: 24°C  
France: 18°C


In [197]:
#A2 topshiriq.

#funksiyalar:
def get_user_city(user_id: str) -> str:
    users_by_city = {
        "user_42": "Toshkent",
        "user_7": "Nukus"
    }

    return users_by_city.get(user_id, "City not found.")

def get_weather(city: str) -> int:
    weathers = {
        "Samarqand": "3C, Snowing",
        "Nukus": "15C, Storm"
    }

    return weathers.get(city, "Weather not found")

def get_dress_code(rule_id) -> str:
    rules = {
        12: "Qishki kurtka va KO’K sharf",
        7: "Kurtka",
        3: "Yengil kiyim"
    }

    return rules.get(int(rule_id), "Not found")


In [200]:
print(get_user_city("user_490"))
print(get_weather("Nukus"))
print(get_dress_code(3))

City not found.
15C, Storm
Yengil kiyim


In [201]:
TOOLS = {
    "get_user_city": get_user_city,
    "get_weather": get_weather,
    "get_dress_code": get_dress_code
    }

In [202]:
llm_tools = [
    {
        "type": "function",
        "name": "get_user_city",
        "description": "Get user city by id",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string"}
            },
            "required": ["user_id"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "get_weather",
        "description": "get weather by city name. name should be in upper case of first letter",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            },
            "required": ["city"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "get_dress_code",
        "description": "get dress code rule by rule id. here are rule ids: [12: while snowing ; 7: while under 10C ; Other conditions are rule 3]",
        "parameters": {
            "type": "object",
            "properties": {
                "rule_id": {"type": "integer"}
            },
            "required": ["rule_id"],
            "additionalProperties": False
        },
        "strict": True
    },

]

In [ ]:
def run_agent(question: str, max_steps=6):
    history = []
    stop = False
    
    history.append({"role": "user", "content": question})

    while not stop and max_steps != 0:
        max_steps -= 1
        res = client.responses.create(
            model=MODEL,
            input=history,
            tools=llm_tools
        )
        history += res.output
        for item in res.output:
            if item.type == "function_call":
                function_name = item.name
                if function_name not in TOOLS:
                    stop=True
                    print(f"Function not found: {item.name}")
                    break

                args = json.loads(item.arguments)
                result = TOOLS[function_name](**args)
                
                print(f"Called: {function_name} | Result: {result}")

                history.append({
                    "type": "function_call_output",
                    "output": result,
                    "call_id": item.call_id
                })
            elif item.type == "message":
                print(item.content[0].text)
                stop = True

        





def ask(question: str):
    run_agent(question)

In [205]:
ask("Men user_12 man. Bugun ishga nima kiyib boray?")

Called: get_user_city | Result: City not found.
user_12 uchun shahar ma’lumoti topilmadi. Qaysi shahardasiz? Shaharni aytsangiz, bugungi ob-havoga qarab ishga nima kiyishni tavsiya qilaman.
